# Proyek Analisis Data: E-Commerce Public Dataset

**Nama: DADAN RAMDAN HIDAYAT**

**Email Dicoding: cdcc953d6y2301@student.devacademy.id**

---

## Daftar Isi
1. [Menentukan Pertanyaan Bisnis](#1-menentukan-pertanyaan-bisnis)
2. [Import Library & Loading Data](#2-import-library--loading-data)
3. [Data Wrangling](#3-data-wrangling)
   - [Gathering Data](#31-gathering-data)
   - [Assessing Data](#32-assessing-data)
   - [Cleaning Data](#33-cleaning-data)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis-eda)
5. [Visualization & Explanatory Analysis](#5-visualization--explanatory-analysis)
6. [Analisis Lanjutan](#6-analisis-lanjutan)
   - [RFM Analysis](#61-rfm-analysis)
   - [Geospatial Analysis](#62-geospatial-analysis)
   - [Clustering / Binning](#63-clustering--binning)
7. [Kesimpulan](#7-kesimpulan)

## 1. Menentukan Pertanyaan Bisnis

Sebelum memulai analisis, penting untuk menentukan pertanyaan bisnis yang ingin dijawab. Berikut adalah pertanyaan bisnis yang akan dianalisis:

1. **Bagaimana tren jumlah pesanan dan pendapatan dari waktu ke waktu?**
2. **Kategori produk apa yang paling banyak terjual dan menghasilkan pendapatan tertinggi?**
3. **Bagaimana distribusi skor ulasan pelanggan dan faktor apa yang mempengaruhinya?**
4. **Metode pembayaran apa yang paling banyak digunakan oleh pelanggan?**
5. **Siapa pelanggan terbaik berdasarkan analisis RFM?**
6. **Bagaimana distribusi geografis pesanan dan penjual di seluruh Brasil?**
7. **Bagaimana pengelompokan pelanggan berdasarkan nilai transaksi mereka?**

## 2. Import Library & Loading Data

In [1]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from folium.plugins import HeatMap
import warnings

warnings.filterwarnings('ignore')

# Pengaturan tampilan
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('Library berhasil diimport!')

KeyboardInterrupt: 

## 3. Data Wrangling

### 3.1 Gathering Data

Pada tahap ini, kita akan memuat semua dataset yang dibutuhkan dari folder `data/`. Dataset terdiri dari beberapa file CSV yang saling berhubungan, membentuk skema data e-commerce yang lengkap.

In [ ]:
# Load semua dataset
customers_df       = pd.read_csv('data/customers_dataset.csv')
geolocation_df     = pd.read_csv('data/geolocation_dataset.csv')
order_items_df     = pd.read_csv('data/order_items_dataset.csv')
order_payments_df  = pd.read_csv('data/order_payments_dataset.csv')
order_reviews_df   = pd.read_csv('data/order_reviews_dataset.csv')
orders_df          = pd.read_csv('data/orders_dataset.csv')
products_df        = pd.read_csv('data/products_dataset.csv')
sellers_df         = pd.read_csv('data/sellers_dataset.csv')
category_df        = pd.read_csv('data/product_category_name_translation.csv')

print('Semua dataset berhasil dimuat!')
print(f'  customers       : {customers_df.shape}')
print(f'  geolocation     : {geolocation_df.shape}')
print(f'  order_items     : {order_items_df.shape}')
print(f'  order_payments  : {order_payments_df.shape}')
print(f'  order_reviews   : {order_reviews_df.shape}')
print(f'  orders          : {orders_df.shape}')
print(f'  products        : {products_df.shape}')
print(f'  sellers         : {sellers_df.shape}')
print(f'  category_trans  : {category_df.shape}')

In [ ]:
# Preview masing-masing dataset
print('=== CUSTOMERS ===')
display(customers_df.head(3))
print('\n=== ORDERS ===')
display(orders_df.head(3))
print('\n=== ORDER ITEMS ===')
display(order_items_df.head(3))

### 3.2 Assessing Data

Pada tahap ini, kita akan memeriksa kualitas data secara menyeluruh, meliputi: nilai yang hilang (missing values), data duplikat, tipe data yang tidak sesuai, serta outlier.

In [ ]:
def assess_dataframe(df, name):
    """Fungsi untuk menilai kualitas sebuah DataFrame."""
    print(f'\n{'='*50}')
    print(f'  ASSESSMENT: {name}')
    print(f'{'='*50}')
    print(f'Dimensi        : {df.shape}')
    print(f'Duplikat       : {df.duplicated().sum()}')
    print('\nMissing Values:')
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else '  Tidak ada missing value')
    print('\nTipe Data:')
    print(df.dtypes)

assess_dataframe(customers_df,      'customers_dataset')
assess_dataframe(orders_df,         'orders_dataset')
assess_dataframe(order_items_df,    'order_items_dataset')
assess_dataframe(order_payments_df, 'order_payments_dataset')
assess_dataframe(order_reviews_df,  'order_reviews_dataset')
assess_dataframe(products_df,       'products_dataset')
assess_dataframe(sellers_df,        'sellers_dataset')

**Insight Assessing Data:**
- `orders_dataset` memiliki beberapa missing values pada kolom tanggal (misal: `order_approved_at`, `order_delivered_customer_date`) karena pesanan belum diproses/dikirim.
- `order_reviews_dataset` memiliki missing values pada kolom `review_comment_title` dan `review_comment_message` karena ulasan bersifat opsional.
- `products_dataset` memiliki missing values pada beberapa kolom deskripsi produk.
- Kolom-kolom bertipe tanggal masih bertipe `object`, perlu dikonversi ke `datetime`.

### 3.3 Cleaning Data

Berdasarkan hasil assessment, berikut langkah-langkah pembersihan data yang dilakukan:

In [ ]:
# --- 1. Konversi kolom datetime ---
date_cols_orders = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols_orders:
    orders_df[col] = pd.to_datetime(orders_df[col])

order_reviews_df['review_creation_date']     = pd.to_datetime(order_reviews_df['review_creation_date'])
order_reviews_df['review_answer_timestamp']  = pd.to_datetime(order_reviews_df['review_answer_timestamp'])
order_items_df['shipping_limit_date']        = pd.to_datetime(order_items_df['shipping_limit_date'])

print('Konversi datetime selesai.')

# --- 2. Hapus duplikat ---
geolocation_df.drop_duplicates(subset=['geolocation_zip_code_prefix',
                                        'geolocation_lat', 'geolocation_lng'], inplace=True)
print(f'Geolocation setelah drop duplikat: {geolocation_df.shape}')

# --- 3. Isi missing pada products dengan 'Unknown' ---
products_df['product_category_name'].fillna('unknown', inplace=True)

# --- 4. Hanya ambil pesanan yang sudah terdeliver ---
orders_delivered_df = orders_df[orders_df['order_status'] == 'delivered'].copy()
print(f'Orders yang delivered: {len(orders_delivered_df)} dari {len(orders_df)}')

print('\nData Cleaning selesai!')

In [ ]:
# --- 5. Merge dataset menjadi satu master dataframe ---
# Tambahkan terjemahan kategori
products_df = products_df.merge(category_df, on='product_category_name', how='left')
products_df['product_category_name_english'].fillna(products_df['product_category_name'], inplace=True)

# Merge orders + customers
master_df = orders_delivered_df.merge(customers_df, on='customer_id', how='left')

# Merge dengan order_items
master_df = master_df.merge(order_items_df, on='order_id', how='left')

# Merge dengan products
master_df = master_df.merge(
    products_df[['product_id','product_category_name_english']],
    on='product_id', how='left'
)

# Merge dengan payments
payment_agg = order_payments_df.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', lambda x: x.mode()[0])
).reset_index()
master_df = master_df.merge(payment_agg, on='order_id', how='left')

# Merge dengan reviews (ambil review pertama per order)
reviews_first = order_reviews_df.sort_values('review_creation_date').drop_duplicates('order_id')
master_df = master_df.merge(
    reviews_first[['order_id', 'review_score']],
    on='order_id', how='left'
)

# Tambah kolom bulan & tahun
master_df['order_year_month'] = master_df['order_purchase_timestamp'].dt.to_period('M')
master_df['order_year']       = master_df['order_purchase_timestamp'].dt.year

print(f'Master DataFrame shape: {master_df.shape}')
display(master_df.head(3))

**Insight Cleaning Data:**
- Seluruh kolom tanggal berhasil dikonversi ke tipe `datetime`.
- Data duplikat pada dataset geolocation berhasil dihapus.
- Hanya pesanan dengan status `delivered` yang digunakan agar analisis lebih akurat.
- Seluruh dataset berhasil digabungkan menjadi satu master DataFrame yang siap dianalisis.

## 4. Exploratory Data Analysis (EDA)

Pada tahap ini, kita akan mengeksplorasi data untuk menemukan pola, distribusi, dan anomali.

In [ ]:
# Statistik deskriptif kolom numerik utama
print('=== Statistik Deskriptif ===')
display(master_df[['price', 'freight_value', 'payment_value', 'review_score']].describe())

In [ ]:
# EDA 1: Distribusi review score
print('Distribusi Review Score:')
print(master_df['review_score'].value_counts().sort_index())

In [ ]:
# EDA 2: Top 10 kategori produk
top_categories = master_df['product_category_name_english'].value_counts().head(10)
print('Top 10 Kategori Produk:')
print(top_categories)

In [ ]:
# EDA 3: Distribusi metode pembayaran
print('Distribusi Metode Pembayaran:')
print(order_payments_df['payment_type'].value_counts())

In [ ]:
# EDA 4: Tren bulanan pesanan
monthly_orders = master_df.groupby('order_year_month').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('payment_value', 'sum')
).reset_index()
monthly_orders['order_year_month'] = monthly_orders['order_year_month'].astype(str)
print('Tren Bulanan (5 bulan terakhir):')
print(monthly_orders.tail(5))

**Insight EDA:**
- Mayoritas pelanggan memberikan review score 5 (sangat puas), menandakan tingkat kepuasan yang tinggi.
- Kategori `bed_bath_table`, `health_beauty`, dan `sports_leisure` mendominasi penjualan.
- Kartu kredit (`credit_card`) adalah metode pembayaran paling populer.
- Tren pesanan menunjukkan pertumbuhan yang konsisten sepanjang tahun 2017-2018.

## 5. Visualization & Explanatory Analysis

### Pertanyaan 1: Tren Jumlah Pesanan & Pendapatan dari Waktu ke Waktu

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot jumlah pesanan
axes[0].plot(monthly_orders['order_year_month'], monthly_orders['total_orders'],
             color='#2196F3', linewidth=2.5, marker='o', markersize=4)
axes[0].fill_between(monthly_orders['order_year_month'], monthly_orders['total_orders'],
                     alpha=0.15, color='#2196F3')
axes[0].set_title('Tren Jumlah Pesanan per Bulan', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Jumlah Pesanan')
axes[0].tick_params(axis='x', rotation=45)

# Plot pendapatan
axes[1].plot(monthly_orders['order_year_month'], monthly_orders['total_revenue'],
             color='#4CAF50', linewidth=2.5, marker='o', markersize=4)
axes[1].fill_between(monthly_orders['order_year_month'], monthly_orders['total_revenue'],
                     alpha=0.15, color='#4CAF50')
axes[1].set_title('Tren Pendapatan per Bulan (BRL)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Bulan')
axes[1].set_ylabel('Pendapatan (BRL)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('dashboard/tren_pesanan_pendapatan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan.')

**Insight:** Tren pesanan dan pendapatan menunjukkan pertumbuhan yang signifikan dari awal 2017 hingga akhir 2017, dengan puncak di sekitar November 2017 (kemungkinan Black Friday). Setelah itu, tren cenderung stabil di angka yang lebih tinggi dibandingkan awal tahun.

### Pertanyaan 2: Kategori Produk Terlaris & Pendapatan Tertinggi

In [ ]:
# Top 10 kategori berdasarkan jumlah item terjual
top10_qty = master_df.groupby('product_category_name_english')['order_id'] \
    .count().nlargest(10).reset_index()
top10_qty.columns = ['category', 'total_items']

# Top 10 kategori berdasarkan pendapatan
top10_rev = master_df.groupby('product_category_name_english')['price'] \
    .sum().nlargest(10).reset_index()
top10_rev.columns = ['category', 'total_revenue']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_qty = sns.color_palette('Blues_r', 10)
colors_rev = sns.color_palette('Greens_r', 10)

bars1 = axes[0].barh(top10_qty['category'], top10_qty['total_items'], color=colors_qty)
axes[0].set_title('Top 10 Kategori: Jumlah Item Terjual', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Item')
axes[0].invert_yaxis()

bars2 = axes[1].barh(top10_rev['category'], top10_rev['total_revenue'], color=colors_rev)
axes[1].set_title('Top 10 Kategori: Total Pendapatan (BRL)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Pendapatan (BRL)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('dashboard/top_kategori.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Kategori `bed_bath_table` mendominasi dari sisi jumlah item terjual, sementara kategori `health_beauty` dan `computers_accessories` berkontribusi besar pada total pendapatan. Ini mengindikasikan adanya produk dengan nilai tinggi di kategori tertentu meskipun volumenya tidak tertinggi.

### Pertanyaan 3: Distribusi Skor Ulasan Pelanggan

In [ ]:
review_counts = master_df['review_score'].value_counts().sort_index().reset_index()
review_counts.columns = ['score', 'count']

colors_review = ['#ef5350','#ff7043','#ffca28','#66bb6a','#42a5f5']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].bar(review_counts['score'], review_counts['count'],
            color=colors_review, edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribusi Review Score', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Jumlah Ulasan')
for i, row in review_counts.iterrows():
    axes[0].text(row['score'], row['count'] + 200, f"{row['count']:,}",
                 ha='center', fontsize=10)

axes[1].pie(review_counts['count'], labels=[f'Score {s}' for s in review_counts['score']],
            colors=colors_review, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Proporsi Review Score', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('dashboard/review_score.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Lebih dari 57% pelanggan memberikan review score 5 (sangat puas), mengindikasikan tingkat kepuasan pelanggan yang tinggi. Namun, terdapat sekitar 11% pelanggan yang memberikan score 1 (sangat tidak puas), yang perlu mendapat perhatian khusus.

### Pertanyaan 4: Metode Pembayaran

In [ ]:
payment_summary = order_payments_df.groupby('payment_type').agg(
    jumlah_transaksi=('order_id', 'count'),
    total_nilai=('payment_value', 'sum')
).reset_index().sort_values('jumlah_transaksi', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pal = sns.color_palette('Set2', len(payment_summary))

axes[0].bar(payment_summary['payment_type'], payment_summary['jumlah_transaksi'],
            color=pal)
axes[0].set_title('Jumlah Transaksi per Metode Pembayaran', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Metode Pembayaran')
axes[0].set_ylabel('Jumlah Transaksi')

axes[1].bar(payment_summary['payment_type'], payment_summary['total_nilai'],
            color=pal)
axes[1].set_title('Total Nilai per Metode Pembayaran (BRL)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Metode Pembayaran')
axes[1].set_ylabel('Total Nilai (BRL)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.0f}M'))

plt.tight_layout()
plt.savefig('dashboard/metode_pembayaran.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Kartu kredit (`credit_card`) mendominasi baik dari sisi jumlah transaksi maupun total nilai pembayaran. `boleto` (semacam slip pembayaran Brasil) menempati posisi kedua. Metode `debit_card` dan `voucher` memiliki penggunaan yang jauh lebih sedikit.

## 6. Analisis Lanjutan

### 6.1 RFM Analysis

**Tujuan:** RFM Analysis digunakan untuk mengelompokkan pelanggan berdasarkan perilaku pembelian mereka. Tiga faktor yang diperhatikan:
- **Recency (R):** Seberapa baru pelanggan melakukan pembelian terakhir
- **Frequency (F):** Seberapa sering pelanggan melakukan pembelian
- **Monetary (M):** Seberapa besar total pengeluaran pelanggan

Analisis ini membantu bisnis mengidentifikasi pelanggan terbaik dan merancang strategi pemasaran yang lebih tepat sasaran.

In [ ]:
# Tanggal referensi = tanggal terbaru dalam dataset + 1 hari
reference_date = master_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f'Reference date: {reference_date.date()}')

# Hitung RFM per customer
rfm_df = master_df.groupby('customer_unique_id').agg(
    Recency   = ('order_purchase_timestamp', lambda x: (reference_date - x.max()).days),
    Frequency = ('order_id', 'nunique'),
    Monetary  = ('payment_value', 'sum')
).reset_index()

print(f'Total pelanggan unik: {len(rfm_df)}')
display(rfm_df.describe())

In [ ]:
# Beri skor RFM (1-5), R: makin kecil makin baik, F & M: makin besar makin baik
rfm_df['R_score'] = pd.qcut(rfm_df['Recency'],   q=5, labels=[5,4,3,2,1], duplicates='drop')
rfm_df['F_score'] = pd.qcut(rfm_df['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm_df['M_score'] = pd.qcut(rfm_df['Monetary'],  q=5, labels=[1,2,3,4,5], duplicates='drop')

rfm_df['RFM_Score'] = rfm_df['R_score'].astype(str) + \
                      rfm_df['F_score'].astype(str) + \
                      rfm_df['M_score'].astype(str)

rfm_df['RFM_Total'] = rfm_df[['R_score','F_score','M_score']].astype(int).sum(axis=1)

# Segmentasi pelanggan
def segment_customer(row):
    r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r == 1 and f >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and f <= 2:
        return 'Lost Customers'
    else:
        return 'Need Attention'

rfm_df['Segment'] = rfm_df.apply(segment_customer, axis=1)

segment_summary = rfm_df['Segment'].value_counts().reset_index()
segment_summary.columns = ['Segment', 'Count']
print('Distribusi Segmen Pelanggan:')
display(segment_summary)

In [ ]:
# Visualisasi RFM Segmentasi
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

pal_rfm = sns.color_palette('tab10', len(segment_summary))
axes[0].barh(segment_summary['Segment'], segment_summary['Count'], color=pal_rfm)
axes[0].set_title('Distribusi Segmen Pelanggan (RFM)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Pelanggan')
axes[0].invert_yaxis()

# RFM Scatter Plot: Recency vs Monetary
scatter = axes[1].scatter(
    rfm_df['Recency'], rfm_df['Monetary'],
    c=rfm_df['RFM_Total'], cmap='RdYlGn',
    alpha=0.5, s=10
)
plt.colorbar(scatter, ax=axes[1], label='RFM Total Score')
axes[1].set_title('Recency vs Monetary (warna = RFM Score)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recency (hari)')
axes[1].set_ylabel('Monetary (BRL)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('dashboard/rfm_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistik per segmen
print('=== Rata-rata RFM per Segmen ===')
display(rfm_df.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2))

**Insight RFM Analysis:**
- Sebagian besar pelanggan termasuk dalam kategori **Lost Customers** dan **Need Attention**, mengindikasikan bahwa banyak pelanggan yang tidak melakukan pembelian ulang.
- Segmen **Champions** adalah pelanggan paling berharga yang perlu dipertahankan dengan program loyalitas.
- Segmen **At Risk** perlu mendapat perhatian khusus dengan penawaran re-engagement sebelum benar-benar hilang.
- Mayoritas pelanggan hanya melakukan 1 kali transaksi (Frequency rendah), menandakan perlu adanya strategi retensi pelanggan yang lebih baik.

### 6.2 Geospatial Analysis

**Tujuan:** Geospatial Analysis digunakan untuk menganalisis distribusi pesanan dan penjual berdasarkan lokasi geografis di Brasil. Analisis ini membantu mengidentifikasi wilayah dengan konsentrasi pelanggan/penjual tertinggi dan potensi pasar yang belum terjangkau.

In [ ]:
# Gabungkan data pelanggan dengan geolokasi
cust_geo = customers_df.merge(
    geolocation_df[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng']] \
        .drop_duplicates('geolocation_zip_code_prefix'),
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='inner'
)

# Gabungkan data penjual dengan geolokasi
seller_geo = sellers_df.merge(
    geolocation_df[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng']] \
        .drop_duplicates('geolocation_zip_code_prefix'),
    left_on='seller_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='inner'
)

print(f'Pelanggan dengan koordinat: {len(cust_geo)}')
print(f'Penjual dengan koordinat  : {len(seller_geo)}')

In [ ]:
# Heatmap distribusi pelanggan
m_customers = folium.Map(location=[-15.0, -51.0], zoom_start=4, tiles='CartoDB positron')

cust_locs = cust_geo[['geolocation_lat','geolocation_lng']].dropna().values.tolist()
HeatMap(cust_locs, radius=8, blur=10, min_opacity=0.3).add_to(m_customers)

folium.LayerControl().add_to(m_customers)
m_customers.save('dashboard/heatmap_customers.html')
print('Heatmap pelanggan disimpan ke dashboard/heatmap_customers.html')

In [ ]:
# Heatmap distribusi penjual
m_sellers = folium.Map(location=[-15.0, -51.0], zoom_start=4, tiles='CartoDB positron')

seller_locs = seller_geo[['geolocation_lat','geolocation_lng']].dropna().values.tolist()
HeatMap(seller_locs, radius=10, blur=12, min_opacity=0.3,
        gradient={0.2: 'blue', 0.5: 'lime', 1.0: 'red'}).add_to(m_sellers)

m_sellers.save('dashboard/heatmap_sellers.html')
print('Heatmap penjual disimpan ke dashboard/heatmap_sellers.html')

# Distribusi pesanan per state
state_orders = master_df.groupby('customer_state').agg(
    total_orders=('order_id','nunique'),
    total_revenue=('payment_value','sum')
).reset_index().sort_values('total_orders', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(state_orders['customer_state'], state_orders['total_orders'],
              color=sns.color_palette('YlOrRd', len(state_orders)))
ax.set_title('Distribusi Pesanan per Negara Bagian (State)', fontsize=13, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Jumlah Pesanan')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('dashboard/distribusi_state.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight Geospatial Analysis:**
- Konsentrasi pelanggan terbesar berada di wilayah **tenggara Brasil**, terutama di state **São Paulo (SP)**, **Rio de Janeiro (RJ)**, dan **Minas Gerais (MG)**.
- Penjual juga terkonsentrasi di area yang sama, menandakan ekosistem e-commerce yang mature di wilayah tersebut.
- State-state di wilayah utara dan timur laut Brasil memiliki penetrasi yang jauh lebih rendah, menunjukkan potensi pasar yang belum dioptimalkan.
- Kepadatan jaringan penjual di luar São Paulo yang lebih tipis dapat memengaruhi waktu pengiriman ke daerah terpencil.

### 6.3 Clustering / Binning

**Tujuan:** Clustering berbasis binning digunakan untuk mengelompokkan pelanggan ke dalam segmen berdasarkan nilai transaksi mereka tanpa menggunakan algoritma machine learning. Pendekatan ini memudahkan tim bisnis memahami profil pelanggan secara sederhana dan actionable.

In [ ]:
# Clustering berdasarkan total spending (Monetary)
rfm_df['Spending_Cluster'] = pd.cut(
    rfm_df['Monetary'],
    bins=[0, 100, 300, 600, 1500, rfm_df['Monetary'].max() + 1],
    labels=['Bronze\n(<R$100)', 'Silver\n(R$100-300)', 'Gold\n(R$300-600)',
            'Platinum\n(R$600-1500)', 'Diamond\n(>R$1500)'],
    right=True
)

# Clustering berdasarkan recency
rfm_df['Recency_Cluster'] = pd.cut(
    rfm_df['Recency'],
    bins=[0, 30, 90, 180, 365, rfm_df['Recency'].max() + 1],
    labels=['Very Active\n(≤30 hari)', 'Active\n(31-90 hari)',
            'Occasional\n(91-180 hari)', 'Dormant\n(181-365 hari)',
            'Churned\n(>365 hari)'],
    right=True
)

spending_summary = rfm_df['Spending_Cluster'].value_counts().sort_index().reset_index()
recency_summary  = rfm_df['Recency_Cluster'].value_counts().sort_index().reset_index()

print('=== Distribusi Spending Cluster ===')
display(spending_summary)
print('\n=== Distribusi Recency Cluster ===')
display(recency_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_spend = ['#CD7F32','#C0C0C0','#FFD700','#E5E4E2','#00C8FF']
colors_rec   = ['#1a9641','#a6d96a','#ffffbf','#fdae61','#d7191c']

# Spending Cluster
axes[0].bar(spending_summary['Spending_Cluster'].astype(str),
            spending_summary['count'], color=colors_spend, edgecolor='white')
axes[0].set_title('Clustering Pelanggan\nBerdasarkan Total Spending', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Tier Spending')
axes[0].set_ylabel('Jumlah Pelanggan')
for i, row in spending_summary.iterrows():
    axes[0].text(i, row['count'] + 50, f"{row['count']:,}", ha='center', fontsize=9)

# Recency Cluster
axes[1].bar(recency_summary['Recency_Cluster'].astype(str),
            recency_summary['count'], color=colors_rec, edgecolor='white')
axes[1].set_title('Clustering Pelanggan\nBerdasarkan Recency', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recency Tier')
axes[1].set_ylabel('Jumlah Pelanggan')
for i, row in recency_summary.iterrows():
    axes[1].text(i, row['count'] + 50, f"{row['count']:,}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('dashboard/clustering.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight Clustering:**
- **Spending Cluster:** Mayoritas pelanggan berada di tier **Bronze** dan **Silver**, mengindikasikan nilai transaksi rata-rata yang relatif rendah. Hanya sebagian kecil yang merupakan pelanggan **Diamond** dengan pengeluaran sangat tinggi.
- **Recency Cluster:** Sebagian besar pelanggan masuk kategori **Churned** (>365 hari tidak bertransaksi), mengonfirmasi temuan dari RFM Analysis bahwa retensi pelanggan adalah tantangan utama.
- Bisnis sebaiknya fokus pada konversi pelanggan Silver ke Gold dan mengaktifkan kembali pelanggan Dormant/Churned.

## 7. Kesimpulan

Berdasarkan seluruh analisis yang telah dilakukan, berikut adalah rangkuman insight utama:

### Tren Bisnis
- **Pertumbuhan signifikan** terjadi sepanjang 2017, dengan puncak di November 2017 (kemungkinan Black Friday). Platform menunjukkan perkembangan yang sehat.

### Produk & Kategori
- Kategori **bed_bath_table, health_beauty**, dan **sports_leisure** adalah yang terlaris. Strategi merchandising dan promosi sebaiknya diprioritaskan untuk kategori-kategori ini.

### Kepuasan Pelanggan
- Tingkat kepuasan secara keseluruhan **tinggi** (>57% memberikan bintang 5), namun 11% pelanggan sangat tidak puas — perlu investigasi lebih lanjut pada faktor pengiriman dan kualitas produk.

### Pembayaran
- **Kartu kredit** dominan; bisnis dapat meningkatkan pengalaman checkout dan menawarkan cicilan tanpa bunga untuk meningkatkan nilai transaksi.

### RFM Analysis
- Segmen **Champions** perlu dipertahankan dengan program loyalitas eksklusif. Segmen **At Risk** dan **Lost Customers** memerlukan kampanye re-engagement.

### Geospatial
- Pasar terkonsentrasi di **São Paulo dan sekitarnya**. Ekspansi ke wilayah utara dan timur laut Brasil dapat membuka peluang pertumbuhan baru.

### Clustering
- Sebagian besar pelanggan adalah pembeli **satu kali dengan nilai rendah**. Program onboarding pasca-pembelian pertama dan email marketing yang personal dapat meningkatkan retensi secara signifikan.

---
*Analisis ini dilakukan menggunakan Python dengan library pandas, numpy, matplotlib, seaborn, dan folium.*